# Call a deployed model using the SDK

## Set up the Python environment

A virtual environment keeps this project’s packages separate from Homebrew and other Python projects. This avoids changing the system Python, prevents package-version conflicts, and makes the dependencies reproducible from `requirements.txt`.

Run these commands in a terminal from the project folder. Python 3.14 must already be installed. If `.venv` already exists, skip the first command.

```bash
python3.14 -m venv .venv
source .venv/bin/activate
python --version  # should show Python 3.14.x
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
```

The first command creates `.venv`; activation makes `python` and `pip` refer to that environment. The install command reads the project’s package list. You only need to create the environment once. After opening a new terminal, activate it again before running Python commands.

In VS Code, open the notebook kernel picker and choose **Python (AZ AI-200)** or select the interpreter at `.venv/bin/python`. The notebook must use this environment for its imports.

### Packages in `requirements.txt`

| Package | Why it is needed |
| --- | --- |
| `openai` | Provides the Responses API client used by the Foundry project client. |
| `azure-ai-projects` | Provides Microsoft’s `AIProjectClient` for Foundry project endpoints. |
| `azure-identity` | Gets an Azure access token using `DefaultAzureCredential`. |
| `python-dotenv` | Loads local settings such as `AZURE_OPENAI_ENDPOINT` from `.env`. |
| `ipykernel` | Lets Jupyter and VS Code run this virtual environment as a notebook kernel. |

Install packages once when creating the environment, or rerun the terminal install command when `requirements.txt` changes. Avoid putting package installation in a cell that runs every time you open or execute the notebook.

Avoid installing with `/opt/homebrew/bin/python3.14 -m pip ... --user`: that targets Homebrew’s externally managed Python and causes the PEP 668 error. Use `python -m pip` only after activating `.venv`, or run the notebook’s `%pip` cell with the project kernel selected.


## If the model call keeps running

Interrupt a stuck cell with the notebook’s **Stop** button, then check the following before running it again:

- Confirm `.env` contains the correct `AZURE_OPENAI_ENDPOINT` for your Azure resource. Keep the file private; `.env` is ignored by Git.
- Make sure the deployment name in the code matches the deployment in your Azure AI project.
- Sign in with an identity available to `DefaultAzureCredential` (for Azure CLI, run `az login` in a terminal) and confirm it has permission to call the deployment.
- Check that your machine can reach the endpoint. The request now has a 45-second timeout and retries are disabled, so connection and service errors should return instead of silently waiting through repeated attempts.


In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

load_dotenv()
endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment_name = "gpt-4.1-Test-Demo"
token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default")

client = OpenAI(
    base_url=endpoint,
    api_key=token_provider,
    timeout=45.0,
    max_retries=0,
)

response = client.responses.create(
    model=deployment_name,
    input="What is the capital of France?",
)

print(response.output_text)


The capital of France is **Paris**.


## Chat with endpoint and API key (no `az login`)

Use this option when you have a Foundry resource key. It calls the OpenAI-compatible Foundry v1 endpoint with the OpenAI Python SDK. Add the key to `.env` as `AZURE_INFERENCE_CREDENTIAL`; do not paste it into the notebook or commit `.env`.

```text
AZURE_INFERENCE_CREDENTIAL=your-foundry-resource-key
```

The endpoint is read from `AZURE_OPENAI_ENDPOINT` and normalized to end in `/openai/v1/`. The resource key and endpoint must belong to the same Foundry resource, and key authentication must be enabled. This cell uses the deployment name configured below.


In [8]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
if not endpoint.endswith("/openai/v1"):
    endpoint += "/openai/v1"

client = OpenAI(
    base_url=endpoint + "/",
    api_key=os.environ["AZURE_INFERENCE_CREDENTIAL"],
    timeout=45.0,
    max_retries=0,
)

response = client.responses.create(
    model="gpt-4.1-Test-Demo",
    input="What is the capital of France?",
)
print(response.output_text)


The capital of France is **Paris**.
